In [1]:
import pandas as pd
import numpy as np
import os

df = pd.read_csv('../data/raw/all_sessions.csv')
df['LapTime_s'] = pd.to_timedelta(df['LapTime']).dt.total_seconds()
print(df.shape)
print(df['Session'].value_counts())

(8058, 39)
Session
FP1    2830
FP2    2060
Q      1724
FP3    1444
Name: count, dtype: int64


In [2]:
results = []

for (year, gp, driver), group in df.groupby(['Year', 'GrandPrix', 'Driver']):
    row = {'Year': year, 'GrandPrix': gp, 'Driver': driver}

    # Practice session features
    for sess in ['FP1', 'FP2', 'FP3']:
        sess_laps = group[group['Session'] == sess]['LapTime_s']
        row[f'{sess}_best'] = sess_laps.min() if not sess_laps.empty else np.nan
        row[f'{sess}_mean'] = sess_laps.mean() if not sess_laps.empty else np.nan
        row[f'{sess}_std']  = sess_laps.std() if not sess_laps.empty else np.nan

    # Track evolution — how much driver improved from FP1 to FP3
    if not np.isnan(row.get('FP1_best', np.nan)) and not np.isnan(row.get('FP3_best', np.nan)):
        row['track_evolution'] = row['FP1_best'] - row['FP3_best']
    else:
        row['track_evolution'] = np.nan

    # Weather features from qualifying
    q_laps = group[group['Session'] == 'Q']
    row['AirTemp']   = q_laps['AirTemp'].mean()
    row['TrackTemp'] = q_laps['TrackTemp'].mean()
    row['Humidity']  = q_laps['Humidity'].mean()
    row['Rainfall']  = int(q_laps['Rainfall'].any()) if not q_laps.empty else 0

    # Team
    row['Team'] = group['Team'].iloc[0] if 'Team' in group.columns else 'Unknown'

    # Target — best qualifying lap time
    q_times = group[group['Session'] == 'Q']['LapTime_s']
    row['quali_best'] = q_times.min() if not q_times.empty else np.nan

    results.append(row)

features_df = pd.DataFrame(results)
features_df = features_df.dropna(subset=['quali_best'])
print(f"Feature table shape: {features_df.shape}")
features_df.head()

Feature table shape: (337, 19)


,Year,GrandPrix,Driver,FP1_best,FP1_mean,FP1_std,FP2_best,FP2_mean,FP2_std,FP3_best,FP3_mean,FP3_std,track_evolution,AirTemp,TrackTemp,Humidity,Rainfall,Team,quali_best
0,2023,Australian Grand Prix,ALB,79.766,80.581667,0.982823,81.182,81.182,NaN,78.553,79.342833,0.938511,1.213,15.204211,22.712632,53.4,1,Williams,77.609
1,2023,Australian Grand Prix,ALO,79.317,82.208889,1.773777,78.887,79.657,1.088944,77.727,79.889875,2.278549,1.590,15.204211,22.712632,53.4,1,Aston Martin,77.139
2,2023,Australian Grand Prix,BOT,80.419,81.691000,0.852589,80.312,81.489,1.496578,78.809,79.906500,0.851495,1.610,15.204211,22.712632,53.4,1,Alfa Romeo,78.714
3,2023,Australian Grand Prix,DEV,79.933,81.472000,1.070387,80.600,80.896,0.503191,79.092,80.080667,1.043198,0.841,15.204211,22.712632,53.4,1,AlphaTauri,78.335
4,2023,Australian Grand Prix,GAS,79.646,81.398200,1.718447,80.206,80.944,0.851820,78.094,79.645714,1.838121,1.552,15.204211,22.712632,53.4,1,Alpine,77.574


In [5]:
from sklearn.preprocessing import LabelEncoder
import joblib
import os

models_dir = '/Users/harshiniratnakumar/Desktop/f1-quali-predictor/models'
os.makedirs(models_dir, exist_ok=True)

for col in ['Driver', 'Team', 'GrandPrix']:
    le = LabelEncoder()
    features_df[f'{col}_enc'] = le.fit_transform(features_df[col].astype(str))
    joblib.dump(le, os.path.join(models_dir, f'le_{col}.pkl'))
    print(f"✅ Saved encoder for {col}")

✅ Saved encoder for Driver
✅ Saved encoder for Team
✅ Saved encoder for GrandPrix


In [4]:
os.makedirs('../data/processed', exist_ok=True)
features_df.to_csv('../data/processed/features.csv', index=False)
print(f"✅ Saved {len(features_df)} rows to data/processed/features.csv")

✅ Saved 337 rows to data/processed/features.csv
